# ML Analysis: 분석모델 vs 예측모델

| | 분석모델 (Part A) | 예측모델 (Part B) |
|---|---|---|
| 목적 | FGI 유의성 / 요인 분석 | CONTRACT_COUNT 예측 정확도 극대화 |
| Lag Feature | **사용 안함** | **사용** |
| 모델 | Linear Regression, Random Forest | Random Forest, XGBoost, LightGBM |

In [ ]:
from snowflake.snowpark.context import get_active_session
import pandas as pd
import numpy as np

session = get_active_session()
session.use_database('PRICE_BASE')
session.use_schema('PUBLIC')

In [ ]:
%%sql -r raw_data
SELECT * FROM PRICE_BASE.PUBLIC.CALC_FGI_INTERNET_NN_MEAN ORDER BY YEAR_MONTH, STATE, CITY

In [ ]:
df = raw_data.copy()
df['YEAR_MONTH'] = pd.to_datetime(df['YEAR_MONTH'])
print(f"Shape: {df.shape}")
print(f"Date Range: {df['YEAR_MONTH'].min().date()} ~ {df['YEAR_MONTH'].max().date()}")
print(f"States: {df['STATE'].nunique()}, Cities: {df['CITY'].nunique()}")
print(f"Null: {df.isnull().sum().sum()}")

---
# PART A: 분석모델
> 목적: CONTRACT_COUNT에 영향을 주는 요인 분석 & FGI 유의성 확인
>
> **Lag feature 사용 금지** — 해석력 유지

In [ ]:
target = 'CONTRACT_COUNT'

features_analysis = [
    'MEAN_MEME_PRICE', 'MEAN_JEONSE_PRICE',
    'CHANGE_MEME_PRICE_RATE', 'REVERSE_JEONSE_PER_MEME', 'RATE_GAP',
    'MOVEMENT_RATE', 'PRICE_MOMENTUM_3M', 'PRICE_VOLATILITY_6M',
    'FEAR_GREED_INDEX'
]

df_a = df.sort_values('YEAR_MONTH').reset_index(drop=True)
split_date = pd.Timestamp('2025-07-01')

train_a = df_a[df_a['YEAR_MONTH'] < split_date]
test_a = df_a[df_a['YEAR_MONTH'] >= split_date]

X_train_a = train_a[features_analysis]
y_train_a = train_a[target]
X_test_a = test_a[features_analysis]
y_test_a = test_a[target]

print(f"[Part A] Features: {len(features_analysis)} (NO Lag)")
print(f"Train: {len(train_a)} rows ({train_a['YEAR_MONTH'].min().date()} ~ {train_a['YEAR_MONTH'].max().date()})")
print(f"Test:  {len(test_a)} rows ({test_a['YEAR_MONTH'].min().date()} ~ {test_a['YEAR_MONTH'].max().date()})")

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

lr = LinearRegression()
lr.fit(X_train_a, y_train_a)
lr_pred = lr.predict(X_test_a)

rf_a = RandomForestRegressor(n_estimators=300, max_depth=15, min_samples_leaf=5, random_state=42, n_jobs=-1)
rf_a.fit(X_train_a, y_train_a)
rf_a_pred = rf_a.predict(X_test_a)

def eval_model(name, y_true, y_pred):
    return {'Model': name, 'MAE': round(mean_absolute_error(y_true, y_pred), 2),
            'RMSE': round(np.sqrt(mean_squared_error(y_true, y_pred)), 2),
            'R2': round(r2_score(y_true, y_pred), 4)}

results_a = pd.DataFrame([eval_model('Linear Regression', y_test_a, lr_pred), eval_model('Random Forest', y_test_a, rf_a_pred)])

print("="*60)
print("   PART A: MODEL PERFORMANCE (Analysis)")
print("="*60)
print(results_a.to_string(index=False))
print("="*60)

In [ ]:
print("="*60)
print("   LINEAR REGRESSION COEFFICIENTS")
print("="*60)
lr_coef = pd.DataFrame({'Feature': features_analysis, 'Coefficient': lr.coef_}).sort_values('Coefficient', key=abs, ascending=False)
print(lr_coef.to_string(index=False))
print(f"\nIntercept: {lr.intercept_:.2f}")

print("\n" + "="*60)
print("   RANDOM FOREST FEATURE IMPORTANCE")
print("="*60)
rf_imp_a = pd.DataFrame({'Feature': features_analysis, 'Importance': rf_a.feature_importances_}).sort_values('Importance', ascending=False)
rf_imp_a['Rank'] = range(1, len(rf_imp_a)+1)
rf_imp_a['Pct'] = (rf_imp_a['Importance'] / rf_imp_a['Importance'].sum() * 100).round(1)
print(rf_imp_a.to_string(index=False))

In [ ]:
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['font.family'] = 'DejaVu Sans'

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

lr_sorted = lr_coef.sort_values('Coefficient', key=abs, ascending=True)
colors_lr = ['#e74c3c' if f == 'FEAR_GREED_INDEX' else ('#2ecc71' if v > 0 else '#3498db') for f, v in zip(lr_sorted['Feature'], lr_sorted['Coefficient'])]
axes[0].barh(lr_sorted['Feature'], lr_sorted['Coefficient'], color=colors_lr)
axes[0].set_title('Linear Regression Coefficients', fontsize=13, fontweight='bold')
axes[0].axvline(x=0, color='black', linewidth=0.5)

rf_sorted = rf_imp_a.sort_values('Importance', ascending=True)
colors_rf = ['#e74c3c' if f == 'FEAR_GREED_INDEX' else '#3498db' for f in rf_sorted['Feature']]
axes[1].barh(rf_sorted['Feature'], rf_sorted['Importance'], color=colors_rf)
axes[1].set_title('Random Forest Feature Importance', fontsize=13, fontweight='bold')

for ax in axes:
    ax.tick_params(axis='y', labelsize=9)

plt.suptitle('PART A: Analysis Model (FGI = red)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
fgi_imp = rf_imp_a[rf_imp_a['Feature']=='FEAR_GREED_INDEX'].iloc[0]
fgi_coef = lr_coef[lr_coef['Feature']=='FEAR_GREED_INDEX']['Coefficient'].values[0]
top1 = rf_imp_a.iloc[0]

print("="*70)
print("   PART A: FGI SIGNIFICANCE ANALYSIS")
print("="*70)
print(f"\n1. FGI in Random Forest:")
print(f"   Rank: {int(fgi_imp['Rank'])}/{len(features_analysis)}")
print(f"   Importance: {fgi_imp['Importance']:.4f} ({fgi_imp['Pct']}%)")
print(f"\n2. FGI in Linear Regression:")
print(f"   Coefficient: {fgi_coef:.6f}")
if fgi_coef > 0:
    print(f"   => FGI 1 unit increase -> CONTRACT_COUNT +{fgi_coef:.2f}")
else:
    print(f"   => FGI 1 unit increase -> CONTRACT_COUNT {fgi_coef:.2f}")

print(f"\n3. Most Important Feature:")
print(f"   {top1['Feature']} (Importance: {top1['Importance']:.4f}, {top1['Pct']}%)")

print(f"\n4. FGI Interpretation:")
if int(fgi_imp['Rank']) <= 3:
    print("   => FGI is a SIGNIFICANT predictor of CONTRACT_COUNT")
    print("   => The composite fear-greed index captures meaningful market sentiment")
elif int(fgi_imp['Rank']) <= 6:
    print("   => FGI has MODERATE predictive power")
    print("   => It contributes but is not among the top drivers")
    print("   => Consider adjusting FGI weights to better capture market dynamics")
else:
    print("   => FGI has LIMITED predictive power")
    print("   => Individual components (price, movement) are more informative than the composite")
    print("   => FGI formula weights need significant revision")

print(f"\n5. FGI vs Components:")
fgi_components = ['CHANGE_MEME_PRICE_RATE', 'REVERSE_JEONSE_PER_MEME', 'RATE_GAP', 'MOVEMENT_RATE', 'PRICE_MOMENTUM_3M', 'PRICE_VOLATILITY_6M']
comp_imp = rf_imp_a[rf_imp_a['Feature'].isin(fgi_components)]['Importance'].sum()
print(f"   FGI alone:     {fgi_imp['Importance']:.4f}")
print(f"   Components sum: {comp_imp:.4f}")
print(f"   => Components are {comp_imp/fgi_imp['Importance']:.1f}x more informative than FGI composite")

print("\n" + "="*70)
print("   FGI WEIGHT IMPROVEMENT (based on RF importance)")
print("="*70)
rf_comp = rf_imp_a[rf_imp_a['Feature'].isin(fgi_components)].copy()
rf_comp['Current_Weight'] = rf_comp['Feature'].map({
    'CHANGE_MEME_PRICE_RATE': 0.25, 'REVERSE_JEONSE_PER_MEME': 0.25,
    'RATE_GAP': 0.15, 'MOVEMENT_RATE': 0.15,
    'PRICE_MOMENTUM_3M': 0.15, 'PRICE_VOLATILITY_6M': 0.15
})
rf_comp['Suggested_Weight'] = (rf_comp['Importance'] / rf_comp['Importance'].sum()).round(4)
rf_comp = rf_comp[['Feature','Current_Weight','Suggested_Weight']].sort_values('Suggested_Weight', ascending=False)
print(rf_comp.to_string(index=False))

---
# PART B: 예측모델
> 목적: CONTRACT_COUNT 예측 성능 극대화
>
> **Lag feature 포함** — 시계열 자기상관 활용

In [ ]:
df_b = df.sort_values(['STATE', 'CITY', 'YEAR_MONTH']).reset_index(drop=True)

df_b['CONTRACT_COUNT_LAG_1'] = df_b.groupby(['STATE','CITY'])['CONTRACT_COUNT'].shift(1)
df_b['CONTRACT_COUNT_LAG_2'] = df_b.groupby(['STATE','CITY'])['CONTRACT_COUNT'].shift(2)
df_b['CONTRACT_COUNT_LAG_3'] = df_b.groupby(['STATE','CITY'])['CONTRACT_COUNT'].shift(3)
df_b['CONTRACT_COUNT_MA_3'] = df_b.groupby(['STATE','CITY'])['CONTRACT_COUNT'].transform(lambda x: x.rolling(3, min_periods=1).mean().shift(1))
df_b['CONTRACT_COUNT_MA_6'] = df_b.groupby(['STATE','CITY'])['CONTRACT_COUNT'].transform(lambda x: x.rolling(6, min_periods=1).mean().shift(1))
df_b['CONTRACT_COUNT_CHG'] = df_b.groupby(['STATE','CITY'])['CONTRACT_COUNT'].pct_change()
df_b = df_b.replace([np.inf, -np.inf], np.nan).fillna(0)

features_predict = [
    'MEAN_MEME_PRICE', 'MEAN_JEONSE_PRICE',
    'CHANGE_MEME_PRICE_RATE', 'REVERSE_JEONSE_PER_MEME', 'RATE_GAP',
    'MOVEMENT_COUNT', 'MOVEMENT_RATE',
    'PRICE_MOMENTUM_3M', 'PRICE_VOLATILITY_6M', 'FEAR_GREED_INDEX',
    'CONTRACT_COUNT_LAG_1', 'CONTRACT_COUNT_LAG_2', 'CONTRACT_COUNT_LAG_3',
    'CONTRACT_COUNT_MA_3', 'CONTRACT_COUNT_MA_6', 'CONTRACT_COUNT_CHG'
]

df_b = df_b.sort_values('YEAR_MONTH').reset_index(drop=True)
train_b = df_b[df_b['YEAR_MONTH'] < split_date]
test_b = df_b[df_b['YEAR_MONTH'] >= split_date]

X_train_b = train_b[features_predict]
y_train_b = train_b[target]
X_test_b = test_b[features_predict]
y_test_b = test_b[target]

print(f"[Part B] Features: {len(features_predict)} (WITH Lag)")
print(f"Train: {len(train_b)} | Test: {len(test_b)}")
print(f"\nNew features: CONTRACT_COUNT_LAG_1/2/3, MA_3/6, CHG")

In [ ]:
!pip install xgboost lightgbm -q

In [ ]:
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor

rf_b = RandomForestRegressor(n_estimators=300, max_depth=20, min_samples_leaf=5, random_state=42, n_jobs=-1)
rf_b.fit(X_train_b, y_train_b)
rf_b_pred = rf_b.predict(X_test_b)
rf_b_train_pred = rf_b.predict(X_train_b)

xgb_b = XGBRegressor(n_estimators=300, max_depth=10, learning_rate=0.05, subsample=0.8, colsample_bytree=0.8, random_state=42)
xgb_b.fit(X_train_b, y_train_b)
xgb_b_pred = xgb_b.predict(X_test_b)
xgb_b_train_pred = xgb_b.predict(X_train_b)

lgbm_b = LGBMRegressor(n_estimators=300, max_depth=10, learning_rate=0.05, subsample=0.8, colsample_bytree=0.8, random_state=42, verbose=-1)
lgbm_b.fit(X_train_b, y_train_b)
lgbm_b_pred = lgbm_b.predict(X_test_b)
lgbm_b_train_pred = lgbm_b.predict(X_train_b)

print("Part B: All models trained.")

In [ ]:
test_results = pd.DataFrame([
    eval_model('Random Forest', y_test_b, rf_b_pred),
    eval_model('XGBoost', y_test_b, xgb_b_pred),
    eval_model('LightGBM', y_test_b, lgbm_b_pred)
])
train_results = pd.DataFrame([
    eval_model('Random Forest', y_train_b, rf_b_train_pred),
    eval_model('XGBoost', y_train_b, xgb_b_train_pred),
    eval_model('LightGBM', y_train_b, lgbm_b_train_pred)
])

print("="*65)
print("   PART B: TEST SET PERFORMANCE")
print("="*65)
print(test_results.to_string(index=False))

print("\n" + "="*65)
print("   PART B: TRAIN SET PERFORMANCE (Overfit Check)")
print("="*65)
print(train_results.to_string(index=False))

print("\n--- Overfit Gap (Train R2 - Test R2) ---")
for i, m in enumerate(['RF','XGB','LGBM']):
    gap = train_results.iloc[i]['R2'] - test_results.iloc[i]['R2']
    status = 'OK' if gap < 0.15 else 'WARNING'
    print(f"  {m}: {gap:.4f} ({status})")

best_b = test_results.loc[test_results['RMSE'].idxmin()]
print(f"\nBest Model: {best_b['Model']} (RMSE={best_b['RMSE']}, R2={best_b['R2']})")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(20, 8))

for ax, model, name, color in zip(axes,
    [rf_b, xgb_b, lgbm_b],
    ['Random Forest', 'XGBoost', 'LightGBM'],
    ['#3498db', '#e67e22', '#9b59b6']):
    imp = pd.DataFrame({'Feature': features_predict, 'Importance': model.feature_importances_}).sort_values('Importance', ascending=True)
    colors = ['#e74c3c' if 'LAG' in f or 'MA_' in f or 'CHG' in f else color for f in imp['Feature']]
    ax.barh(imp['Feature'], imp['Importance'], color=colors)
    ax.set_title(name, fontsize=13, fontweight='bold')
    ax.tick_params(axis='y', labelsize=8)

plt.suptitle('PART B: Prediction Model Feature Importance (Lag/MA/CHG = red)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
lag_feats = ['CONTRACT_COUNT_LAG_1','CONTRACT_COUNT_LAG_2','CONTRACT_COUNT_LAG_3','CONTRACT_COUNT_MA_3','CONTRACT_COUNT_MA_6','CONTRACT_COUNT_CHG']

print("="*70)
print("   PART B: LAG FEATURE IMPACT ANALYSIS")
print("="*70)
for model, name in [(rf_b,'RF'), (xgb_b,'XGB'), (lgbm_b,'LGBM')]:
    imp = pd.DataFrame({'Feature': features_predict, 'Importance': model.feature_importances_})
    lag_imp = imp[imp['Feature'].isin(lag_feats)]['Importance'].sum()
    total = imp['Importance'].sum()
    print(f"\n  {name}: Lag features contribute {lag_imp/total*100:.1f}% of total importance")
    top3 = imp.sort_values('Importance', ascending=False).head(3)
    for _, r in top3.iterrows():
        print(f"    {r['Feature']:30s}: {r['Importance']:.4f}")

print("\n" + "="*70)
print("   FINAL SUMMARY")
print("="*70)
print(f"\n[Part A - Analysis Model]")
print(f"  Purpose: Factor analysis & FGI significance")
print(f"  Best: Random Forest (R2={results_a.iloc[1]['R2']})")
print(f"  FGI Rank: {int(fgi_imp['Rank'])}/{len(features_analysis)}")
print(f"  Top Feature: {top1['Feature']}")

print(f"\n[Part B - Prediction Model]")
print(f"  Purpose: Maximize prediction accuracy")
print(f"  Best: {best_b['Model']} (R2={best_b['R2']}, RMSE={best_b['RMSE']})")
print(f"  Key Driver: CONTRACT_COUNT_LAG_1 (autoregressive)")

print(f"\n[Key Insight]")
print(f"  Part A tells you WHY contracts change (market factors)")
print(f"  Part B tells you HOW MANY contracts to expect (time series)")

---
## Part A-2: FGI 단독 유의성 검증
> FGI 구성 요소와 함께 넣으면 다중공선성으로 FGI가 묻힘
>
> → FGI만 단독으로 넣어서 설명력 확인

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

def eval_model(name, y_true, y_pred):
    return {'Model': name, 'MAE': round(mean_absolute_error(y_true, y_pred), 2),
            'RMSE': round(np.sqrt(mean_squared_error(y_true, y_pred)), 2),
            'R2': round(r2_score(y_true, y_pred), 4)}

results_list = []

lr_fgi = LinearRegression()
lr_fgi.fit(X_train_a[['FEAR_GREED_INDEX']], y_train_a)
lr_fgi_pred = lr_fgi.predict(X_test_a[['FEAR_GREED_INDEX']])
results_list.append(eval_model('FGI Only (LR)', y_test_a, lr_fgi_pred))

rf_fgi = RandomForestRegressor(n_estimators=300, max_depth=15, min_samples_leaf=5, random_state=42, n_jobs=-1)
rf_fgi.fit(X_train_a[['FEAR_GREED_INDEX']], y_train_a)
rf_fgi_pred = rf_fgi.predict(X_test_a[['FEAR_GREED_INDEX']])
results_list.append(eval_model('FGI Only (RF)', y_test_a, rf_fgi_pred))

no_fgi_feats = [f for f in features_analysis if f != 'FEAR_GREED_INDEX']
lr_no = LinearRegression()
lr_no.fit(X_train_a[no_fgi_feats], y_train_a)
lr_no_pred = lr_no.predict(X_test_a[no_fgi_feats])
results_list.append(eval_model('Without FGI (LR)', y_test_a, lr_no_pred))

rf_no = RandomForestRegressor(n_estimators=300, max_depth=15, min_samples_leaf=5, random_state=42, n_jobs=-1)
rf_no.fit(X_train_a[no_fgi_feats], y_train_a)
rf_no_pred = rf_no.predict(X_test_a[no_fgi_feats])
results_list.append(eval_model('Without FGI (RF)', y_test_a, rf_no_pred))

results_list.append({'Model': 'All Features (LR)', 'MAE': 360.98, 'RMSE': 563.37, 'R2': 0.0213})
results_list.append({'Model': 'All Features (RF)', 'MAE': 269.04, 'RMSE': 470.00, 'R2': 0.3188})

comp = pd.DataFrame(results_list)

print("="*70)
print("   FGI SIGNIFICANCE: ISOLATED COMPARISON")
print("="*70)
print(comp.to_string(index=False))

print("\n" + "="*70)
print("   INTERPRETATION")
print("="*70)

fgi_only_r2 = comp[comp['Model']=='FGI Only (RF)']['R2'].values[0]
no_fgi_r2 = comp[comp['Model']=='Without FGI (RF)']['R2'].values[0]
all_r2 = 0.3188
fgi_lr_r2 = comp[comp['Model']=='FGI Only (LR)']['R2'].values[0]

print(f"\n[RF 기준 R2 비교]")
print(f"  1) FGI 단독:        {fgi_only_r2:.4f}  <- FGI 하나만 사용")
print(f"  2) FGI 제외:        {no_fgi_r2:.4f}  <- 구성 요소만 사용")
print(f"  3) 전체 (FGI+구성요소): {all_r2:.4f}  <- 모두 사용")

print(f"\n[핵심 해석]")
print(f"  FGI 단독 R2 = {fgi_only_r2:.4f} (음수 = 평균값보다 예측이 나쁨)")
print(f"  => FGI 하나만으로는 CONTRACT_COUNT를 전혀 설명하지 못함")
print(f"")
print(f"  FGI 추가 효과: R2 {no_fgi_r2:.4f} -> {all_r2:.4f} (delta: {all_r2 - no_fgi_r2:+.4f})")
print(f"  => FGI를 추가해도 R2가 거의 변하지 않음")

print(f"\n[왜 FGI가 단독으로 작동하지 않는가?]")
print(f"  FGI = (CHANGE_MEME_PRICE_RATE*0.25 + REVERSE_JEONSE_PER_MEME*0.25 + ...) * 100")
print(f"  FGI는 여러 변수의 가중합이지만:")
print(f"    - 현재 가중치가 실제 중요도와 불일치")
print(f"      (CHANGE_MEME_PRICE_RATE: 가중치 25% vs 실제 importance 5.1%)")
print(f"      (MOVEMENT_RATE: 가중치 15% vs 실제 importance 14.9%)")
print(f"    - 가중합으로 정보가 압축되면서 개별 변수의 비선형 패턴이 손실")
print(f"    - 단일 숫자로 172개 시군구의 다양한 패턴을 포착하기 어려움")

print(f"\n[결론]")
print(f"  1. FGI는 단독 예측 변수로서 유의하지 않음 (R2 < 0)")
print(f"  2. 기존 feature에 추가해도 거의 기여 없음 (delta R2 = +{all_r2 - no_fgi_r2:.4f})")
print(f"  3. 구성 요소들을 개별적으로 사용하는 것이 훨씬 효과적")
print(f"  4. FGI를 유의미하게 만들려면 가중치를 RF importance 기반으로 재설계 필요")

---
# Part C: ML 기반 FGI 재설계

기존 FGI는 선형 가중합 → R² < 0 (무의미)

**새로운 접근**: 비선형 ML 모델로 시장 변수 → CONTRACT_COUNT를 학습한 후,
그 예측값을 정규화하여 ML_FGI (0~100)로 변환

비교 실험 4가지:
1. 기존 선형 FGI 단독
2. 개별 시장 변수
3. ML_FGI 단독
4. 개별 시장 변수 + ML_FGI

In [ ]:
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor

def eval_model(name, y_true, y_pred):
    return {'Model': name, 'MAE': round(mean_absolute_error(y_true, y_pred), 2),
            'RMSE': round(np.sqrt(mean_squared_error(y_true, y_pred)), 2),
            'R2': round(r2_score(y_true, y_pred), 4)}

market_features = [
    'CHANGE_MEME_PRICE_RATE', 'REVERSE_JEONSE_PER_MEME', 'RATE_GAP',
    'MOVEMENT_RATE', 'PRICE_MOMENTUM_3M', 'PRICE_VOLATILITY_6M'
]

all_analysis_features = [
    'MEAN_MEME_PRICE', 'MEAN_JEONSE_PRICE',
    'CHANGE_MEME_PRICE_RATE', 'REVERSE_JEONSE_PER_MEME', 'RATE_GAP',
    'MOVEMENT_RATE', 'PRICE_MOMENTUM_3M', 'PRICE_VOLATILITY_6M'
]

target = 'CONTRACT_COUNT'
split_date = pd.Timestamp('2025-07-01')
df_c = df.sort_values('YEAR_MONTH').reset_index(drop=True)
train_c = df_c[df_c['YEAR_MONTH'] < split_date].copy()
test_c = df_c[df_c['YEAR_MONTH'] >= split_date].copy()

print(f"Market features (FGI components): {market_features}")
print(f"Train: {len(train_c)} | Test: {len(test_c)}")

In [ ]:
ml_fgi_models = {}
ml_fgi_scores = []

for name, Model, params in [
    ('RF', RandomForestRegressor, {'n_estimators': 300, 'max_depth': 12, 'min_samples_leaf': 10, 'random_state': 42, 'n_jobs': -1}),
    ('XGB', XGBRegressor, {'n_estimators': 300, 'max_depth': 8, 'learning_rate': 0.05, 'subsample': 0.8, 'random_state': 42}),
    ('LGBM', LGBMRegressor, {'n_estimators': 300, 'max_depth': 8, 'learning_rate': 0.05, 'subsample': 0.8, 'random_state': 42, 'verbose': -1}),
]:
    m = Model(**params)
    m.fit(train_c[market_features], train_c[target])
    
    train_pred = m.predict(train_c[market_features])
    test_pred = m.predict(test_c[market_features])
    
    all_pred = np.concatenate([train_pred, test_pred])
    p_min, p_max = all_pred.min(), all_pred.max()
    
    train_fgi = (train_pred - p_min) / (p_max - p_min) * 100
    test_fgi = (test_pred - p_min) / (p_max - p_min) * 100
    
    ml_fgi_models[name] = {'model': m, 'min': p_min, 'max': p_max}
    
    train_c[f'ML_FGI_{name}'] = train_fgi
    test_c[f'ML_FGI_{name}'] = test_fgi
    
    lr_check = LinearRegression()
    lr_check.fit(train_fgi.reshape(-1,1), train_c[target])
    lr_pred = lr_check.predict(test_fgi.reshape(-1,1))
    r2_solo = r2_score(test_c[target], lr_pred)
    
    rf_check = RandomForestRegressor(n_estimators=200, max_depth=12, random_state=42, n_jobs=-1)
    rf_check.fit(train_fgi.reshape(-1,1), train_c[target])
    rf_solo_pred = rf_check.predict(test_fgi.reshape(-1,1))
    r2_solo_rf = r2_score(test_c[target], rf_solo_pred)
    
    ml_fgi_scores.append({'Base_Model': name, 'Solo_R2_LR': round(r2_solo, 4), 'Solo_R2_RF': round(r2_solo_rf, 4)})
    print(f"{name}: ML_FGI range [{train_fgi.min():.1f} ~ {train_fgi.max():.1f}], Solo R2(LR)={r2_solo:.4f}, Solo R2(RF)={r2_solo_rf:.4f}")

ml_fgi_df = pd.DataFrame(ml_fgi_scores)
print("\n--- ML_FGI Solo Performance ---")
print(ml_fgi_df.to_string(index=False))

best_base = ml_fgi_df.loc[ml_fgi_df['Solo_R2_RF'].idxmax(), 'Base_Model']
print(f"\nBest ML_FGI base: {best_base}")
ml_fgi_col = f'ML_FGI_{best_base}'

In [ ]:
experiments = []

m1 = RandomForestRegressor(n_estimators=300, max_depth=15, min_samples_leaf=5, random_state=42, n_jobs=-1)
m1.fit(train_c[['FEAR_GREED_INDEX']], train_c[target])
pred1 = m1.predict(test_c[['FEAR_GREED_INDEX']])
experiments.append(eval_model('1. Old FGI Only', test_c[target], pred1))

m2 = RandomForestRegressor(n_estimators=300, max_depth=15, min_samples_leaf=5, random_state=42, n_jobs=-1)
m2.fit(train_c[all_analysis_features], train_c[target])
pred2 = m2.predict(test_c[all_analysis_features])
experiments.append(eval_model('2. Market Variables', test_c[target], pred2))

m3 = RandomForestRegressor(n_estimators=300, max_depth=15, min_samples_leaf=5, random_state=42, n_jobs=-1)
m3.fit(train_c[[ml_fgi_col]], train_c[target])
pred3 = m3.predict(test_c[[ml_fgi_col]])
experiments.append(eval_model('3. ML_FGI Only', test_c[target], pred3))

feats_with_ml_fgi = all_analysis_features + [ml_fgi_col]
m4 = RandomForestRegressor(n_estimators=300, max_depth=15, min_samples_leaf=5, random_state=42, n_jobs=-1)
m4.fit(train_c[feats_with_ml_fgi], train_c[target])
pred4 = m4.predict(test_c[feats_with_ml_fgi])
experiments.append(eval_model('4. Market + ML_FGI', test_c[target], pred4))

exp_df = pd.DataFrame(experiments)

print("="*70)
print("   4-WAY COMPARISON (all using Random Forest)")
print("="*70)
print(exp_df.to_string(index=False))
print("="*70)

old_fgi_r2 = exp_df.iloc[0]['R2']
ml_fgi_r2 = exp_df.iloc[2]['R2']
market_r2 = exp_df.iloc[1]['R2']
market_ml_r2 = exp_df.iloc[3]['R2']

print(f"\n[Key Comparisons]")
print(f"  Old FGI vs ML_FGI (solo):     R2 {old_fgi_r2:.4f} -> {ml_fgi_r2:.4f} (delta: {ml_fgi_r2 - old_fgi_r2:+.4f})")
print(f"  Market vars vs Market+ML_FGI: R2 {market_r2:.4f} -> {market_ml_r2:.4f} (delta: {market_ml_r2 - market_r2:+.4f})")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

colors = ['#e74c3c', '#3498db', '#2ecc71', '#9b59b6']
models = exp_df['Model'].tolist()
r2_vals = exp_df['R2'].tolist()
rmse_vals = exp_df['RMSE'].tolist()

axes[0].barh(models, r2_vals, color=colors)
axes[0].set_title('R² Comparison', fontsize=13, fontweight='bold')
axes[0].set_xlim(min(r2_vals) - 0.05, max(r2_vals) + 0.05)
axes[0].axvline(x=0, color='black', linewidth=0.5)
for i, v in enumerate(r2_vals):
    axes[0].text(v + 0.005, i, f'{v:.4f}', va='center', fontsize=10)

axes[1].barh(models, rmse_vals, color=colors)
axes[1].set_title('RMSE Comparison (lower is better)', fontsize=13, fontweight='bold')
for i, v in enumerate(rmse_vals):
    axes[1].text(v + 2, i, f'{v:.1f}', va='center', fontsize=10)

plt.suptitle(f'Old FGI vs ML_FGI ({best_base}-based) Performance', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
corr_old = train_c[['FEAR_GREED_INDEX', target]].corr().iloc[0,1]
corr_ml = train_c[[ml_fgi_col, target]].corr().iloc[0,1]

print("="*70)
print(f"   FINAL CONCLUSION: ML_FGI ({best_base}-based)")
print("="*70)

print(f"\n[1] Best ML_FGI Base Model: {best_base}")
for _, row in ml_fgi_df.iterrows():
    marker = ' <-- BEST' if row['Base_Model'] == best_base else ''
    print(f"     {row['Base_Model']}: Solo R2(RF)={row['Solo_R2_RF']}{marker}")

print(f"\n[2] Old FGI vs ML_FGI")
print(f"     Old FGI solo R2: {old_fgi_r2:.4f} | Correlation w/ target: {corr_old:.4f}")
print(f"     ML_FGI  solo R2: {ml_fgi_r2:.4f} | Correlation w/ target: {corr_ml:.4f}")
if ml_fgi_r2 > 0:
    print(f"     => ML_FGI가 Old FGI 대비 R2 {ml_fgi_r2 - old_fgi_r2:+.4f} 개선")
    print(f"     => ML_FGI는 단독으로 CONTRACT_COUNT 분산의 {ml_fgi_r2*100:.1f}%를 설명")
else:
    print(f"     => ML_FGI도 단독 설명력 부족")

print(f"\n[3] ML_FGI 단독 유의성")
if ml_fgi_r2 >= 0.15:
    print(f"     => YES - ML_FGI는 단독으로 유의미 (R2={ml_fgi_r2:.4f})")
    print(f"     => 시장 변수 6개를 하나의 지표로 요약 가능")
elif ml_fgi_r2 >= 0.05:
    print(f"     => MODERATE - 약한 설명력 있으나 단독 사용은 한계")
else:
    print(f"     => LIMITED - 단독 설명력 부족, 보조 지표로만 활용")

print(f"\n[4] ML_FGI as 대표 지표?")
print(f"     Market vars R2:          {market_r2:.4f}")
print(f"     ML_FGI solo R2:          {ml_fgi_r2:.4f}")
ratio = ml_fgi_r2 / market_r2 * 100 if market_r2 > 0 else 0
print(f"     ML_FGI는 개별 변수 대비 {ratio:.1f}%의 정보를 보존")
if ratio >= 70:
    print(f"     => ML_FGI가 시장 변수의 대표 지표로 충분히 사용 가능")
elif ratio >= 40:
    print(f"     => ML_FGI가 시장 상황을 상당 부분 요약하나, 세부 분석에는 개별 변수 필요")
else:
    print(f"     => 개별 변수 대비 정보 손실이 큼, 대표 지표로는 부족")

print(f"\n[5] 공포/탐욕 지표로 활용 가능한가?")
print(f"     ML_FGI는 {best_base} 모델이 학습한 '시장 변수 → 계약 활성도' 매핑")
print(f"     높은 ML_FGI = 계약 활발 (탐욕) / 낮은 ML_FGI = 계약 침체 (공포)")
if ml_fgi_r2 >= 0.10:
    print(f"     => 비선형 패턴을 포착하여 시장 심리 대리 변수로 활용 가능")
    print(f"     => 단, 정기적으로 모델 재학습 필요 (시장 구조 변화 반영)")
else:
    print(f"     => 현재 시장 변수만으로는 계약 활성도 예측에 한계")
    print(f"     => 추가 데이터(금리, 공급량, 정책 등) 확보 시 개선 가능")

---
## Part D: Model Registry 등록
Part A (RF 분석모델)과 Part B (XGBoost 예측모델)을 Snowflake Model Registry에 등록합니다.
등록된 모델은 Streamlit 앱에서 실시간 예측에 활용됩니다.

In [ ]:
from snowflake.ml.registry import Registry
from snowflake.ml.model import type_hints as model_types

session.sql('CREATE DATABASE IF NOT EXISTS ML_REGISTRY').collect()
session.sql('CREATE SCHEMA IF NOT EXISTS ML_REGISTRY.PUBLIC').collect()

reg = Registry(session=session, database_name='ML_REGISTRY', schema_name='PUBLIC')
print('Registry initialized: ML_REGISTRY.PUBLIC')

In [ ]:
sample_a = session.create_dataframe(X_test_a.head(10))

model_ref_a = reg.log_model(
    model=rf_a,
    model_name='FGI_ANALYSIS_RF',
    version_name='v1',
    sample_input_data=sample_a,
    conda_dependencies=['scikit-learn'],
    comment='Part A: RF 분석모델 - FGI 기반 시장 요인 분석 (9 features, no lag)',
    metrics={'R2': float(results_a.iloc[1]['R2']), 'RMSE': float(results_a.iloc[1]['RMSE']), 'MAE': float(results_a.iloc[1]['MAE'])}
)
print(f'Part A RF 모델 등록 완료: FGI_ANALYSIS_RF v1')
print(f'Metrics: {model_ref_a.show_metrics()}')

In [ ]:
sample_b = session.create_dataframe(X_test_b.head(10))

model_ref_b = reg.log_model(
    model=xgb_b,
    model_name='FGI_PREDICT_XGB',
    version_name='v1',
    sample_input_data=sample_b,
    conda_dependencies=['scikit-learn', 'xgboost'],
    comment='Part B: XGBoost 예측모델 - 계약건수 예측 (16 features, with lag)',
    metrics={'R2': float(test_results[test_results['Model']=='XGBoost']['R2'].values[0]), 'RMSE': float(test_results[test_results['Model']=='XGBoost']['RMSE'].values[0]), 'MAE': float(test_results[test_results['Model']=='XGBoost']['MAE'].values[0])}
)
print(f'Part B XGBoost 모델 등록 완료: FGI_PREDICT_XGB v1')
print(f'Metrics: {model_ref_b.show_metrics()}')

In [ ]:
models = reg.show_models()
print('등록된 모델 목록:')
print(models[['name', 'comment']].to_string(index=False))

for m in ['FGI_ANALYSIS_RF', 'FGI_PREDICT_XGB']:
    mv = reg.get_model(m).version('v1')
    print(f'\n{m} v1 metrics: {mv.show_metrics()}')
print('\nModel Registry 등록 완료!')